In [181]:
import numpy as np
import pandas as pd

from cvxopt import matrix, solvers
from sklearn.model_selection import train_test_split, ParameterGrid
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw
from sklearn.preprocessing import StandardScaler
from sksurv.linear_model import CoxPHSurvivalAnalysis

from sksurv.util import Surv

def generate_wsvcr_data(n=500, censoring_rate=0.3, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)

    x1 = np.random.normal(1, 1, n)
    x2 = np.random.normal(2, np.sqrt(2), n)
    x3 = np.random.exponential(1, n)

    x2_safe = np.where(np.abs(x2) < 1e-8, 1e-8, x2)

    lam = 5 * np.exp(
        -x1 * np.log(np.abs(x2_safe)) + np.sin(x3 ** 2)
    )

    k = 3
    U = np.random.uniform(0, 1, n)
    Y = lam * (-np.log1p(-U)) ** (1 / k)

    # --- censura controlada ---
    def generate_censoring(Y, target_rate):
        scale = np.median(Y)
        for _ in range(50):
            C = np.random.exponential(scale=scale, size=len(Y))
            delta = (Y <= C).astype(int)
            rate = 1 - delta.mean()

            if abs(rate - target_rate) < 0.01:
                return C

            scale *= 1.2 if rate > target_rate else 0.8

        return C

    C = generate_censoring(Y, censoring_rate)

    T = np.minimum(Y, C)
    delta = (Y <= C).astype(bool)

    l = T.copy()
    u = np.where(delta, T, np.inf)

    return pd.DataFrame({
        "X1": x1,
        "X2": x2,
        "X3": x3,
        "T": T,
        "delta": delta,
        "l": l,
        "u": u
    })

In [169]:
solvers.options['show_progress'] = False

def polynomial_kernel(X, Y=None, degree=3, gamma=1.0, coef0=1.0):
    if Y is None:
        Y = X
    return (gamma * (X @ Y.T) + coef0) ** degree

def wavelet_kernel(X, Y=None, A=1.0):
    if Y is None:
        Y = X
    diff = X[:, None, :] - Y[None, :, :]
    return np.prod(
        np.cos(1.75 * diff / A) * np.exp(-0.5 * (diff / A) ** 2),
        axis=2
    )

def exponential_kernel(X, Y=None, gamma=1.0):
    if Y is None:
        Y = X
    diff = X[:, None, :] - Y[None, :, :]
    dist = np.sqrt(np.sum(diff**2, axis=2))
    return np.exp(-gamma * dist)

def cauchy_kernel(X, Y=None, gamma=1.0):
    if Y is None:
        Y = X
    diff = X[:, None, :] - Y[None, :, :]
    sq_dist = np.sum(diff**2, axis=2)
    return 1 / (1 + gamma * sq_dist)

def rbf_kernel(X, Y=None, gamma=1.0):
    if Y is None:
        Y = X
    diff = X[:, None, :] - Y[None, :, :]
    sq_dist = np.sum(diff**2, axis=2)
    return np.exp(-gamma * sq_dist)

def sigmoid_kernel(X, Y=None, gamma=1.0, coef0=0.0):
    if Y is None:
        Y = X
    return np.tanh(gamma * (X @ Y.T) + coef0)

def laplacian_kernel(X, Y=None, gamma=1.0):
    if Y is None:
        Y = X
    diff = np.abs(X[:, None, :] - Y[None, :, :])
    dist = np.sum(diff, axis=2)
    return np.exp(-gamma * dist)

def linear_kernel(X, Y=None):
    if Y is None:
        Y = X
    return X @ Y.T

class WSVCR_QP:
    def __init__(self, C=1.0, kernel="wavelet", kernel_params=None):
        self.C = C
        self.kernel = kernel
        self.kernel_params = kernel_params if kernel_params else {}

    def _compute_kernel(self, X, Y=None):

        if callable(self.kernel):
            return self.kernel(X, Y, **self.kernel_params)

        if self.kernel == "wavelet":
            return wavelet_kernel(X, Y, **self.kernel_params)

        elif self.kernel in "rbf":
            return rbf_kernel(X, Y, **self.kernel_params)

        elif self.kernel == "linear":
            return linear_kernel(X, Y)

        elif self.kernel == "poly":
            return polynomial_kernel(X, Y, **self.kernel_params)

        elif self.kernel == "sigmoid":
            return sigmoid_kernel(X, Y, **self.kernel_params)

        elif self.kernel == "laplacian":
            return laplacian_kernel(X, Y, **self.kernel_params)

        elif self.kernel == "exponential":
            return exponential_kernel(X, Y, **self.kernel_params)

        elif self.kernel == "cauchy":
            return cauchy_kernel(X, Y, **self.kernel_params)

        else:
            raise ValueError("Unknown kernel")

    def fit(self, X, l, u):
        X = np.asarray(X)
        l = np.asarray(l)
        u = np.asarray(u)

        n = X.shape[0]

        L = np.isfinite(l)
        U = np.isfinite(u)

        K = self._compute_kernel(X)

        P = np.block([
            [K, -K],
            [-K, K]
        ])

        P += 1e-8 * np.eye(2 * n)

        q = np.zeros(2 * n)

        for i in range(n):
            if L[i]:
                q[i] = -l[i]
            if U[i]:
                q[n + i] = u[i]

        A_eq = np.zeros((1, 2 * n))

        for i in range(n):
            if L[i]:
                A_eq[0, i] = 1
            if U[i]:
                A_eq[0, n + i] = -1

        b_eq = np.array([0.0])

        G = np.vstack([
            np.eye(2 * n),
            -np.eye(2 * n)
        ])

        h = np.hstack([
            self.C * np.ones(2 * n),
            np.zeros(2 * n)
        ])

        solution = solvers.qp(
            matrix(P),
            matrix(q),
            matrix(G),
            matrix(h),
            matrix(A_eq),
            matrix(b_eq)
        )

        z = np.array(solution['x']).flatten()

        self.alpha = z[:n]
        self.alpha_star = z[n:]
        self.theta = self.alpha - self.alpha_star

        self.X = X
        self.K = K

        self.b = self.compute_bias(l, u)

    def compute_bias(self, l, u):
        f = self.K @ self.theta

        b_vals = []

        for i in range(len(self.theta)):
            if self.alpha[i] > 1e-6 and np.isfinite(l[i]):
                b_vals.append(l[i] - f[i])
            elif self.alpha_star[i] > 1e-6 and np.isfinite(u[i]):
                b_vals.append(u[i] - f[i])

        return np.mean(b_vals) if b_vals else 0.0

    def predict(self, X_new):
        X_new = np.asarray(X_new)

        K_new = self._compute_kernel(self.X, X_new)

        return self.theta @ K_new + self.b

In [182]:
def cindex_survival(y_time, y_pred, event):
    return concordance_index_censored(
        event,
        y_time,
        -y_pred
    )[0]

def cindex_ipcw_cox(model, X_test, y_train_struct, y_test_struct):
    y_pred = model.predict(X_test)

    return concordance_index_ipcw(
        y_train_struct,
        y_test_struct,
        y_pred
    )[0]


# =========================
# 3. TREINO + VALIDAÇÃO
# =========================

def train_model(model_class, X_train, l_train, u_train, X_val, y_val, d_val, param_grid):
    best_score = -np.inf
    best_model = None

    for params in ParameterGrid(param_grid):
        model = model_class(**params)
        model.fit(X_train, l_train, u_train)

        pred = model.predict(X_val)
        score = cindex_survival(y_val, pred, d_val)

        if score > best_score:
            best_score = score
            best_model = model

    return best_model

In [ ]:
def run_experiment(n, censoring_rate, model_class, param_grid, n_sim=50):

    scores = []

    for _ in range(n_sim):

        # --- gerar novos dados (MONTE CARLO) ---
        df = generate_wsvcr_data(n=n, censoring_rate=censoring_rate)

        X = df[["X1","X2","X3"]].values
        l = df["l"].values
        u = df["u"].values
        T = df["T"].values
        delta = df["delta"].values

        # --- split treino/teste ---
        X_train, X_test, l_train, l_test, u_train, u_test, T_train, T_test, d_train, d_test = train_test_split(
            X, l, u, T, delta,
            test_size=0.3,
            stratify=delta
        )

        # --- split treino/validação ---
        X_tr, X_val, l_tr, l_val, u_tr, u_val, T_tr, T_val, d_tr, d_val = train_test_split(
            X_train, l_train, u_train, T_train, d_train,
            test_size=0.3,
            stratify=d_train
        )

        # --- PADRONIZAÇÃO ---
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)

        best_model = train_model(
            model_class,
            X_tr, l_tr, u_tr,
            X_val, T_val, d_val,
            param_grid
        )

        pred_test = best_model.predict(X_test)
        score = cindex_survival(T_test, pred_test, d_test)

        scores.append(score)

    return np.mean(scores), np.std(scores)

In [ ]:
def run_simul(n, censoring):
    params = [
        {
            "C": [0.25, 0.5, 1, 2, 4, 8],
            "kernel": ["wavelet"],
            "kernel_params": [{"A": A} for A in [0.125, 0.5, 1, 2, 4, 8]]
        },
        {
            "C": [0.25, 0.5, 1, 2, 4, 8],
            "kernel": ["cauchy"],
            "kernel_params": [{"gamma": gamma} for gamma in [0.125, 0.5, 1, 2, 4, 8]]
        },
        {
            "C": [0.25, 0.5, 1, 2, 4, 8],
            "kernel": ["poly"],
            "kernel_params": [{"gamma": gamma} for gamma in [0.125, 0.5, 1, 2, 4, 8]]
        },
        {
            "C": [0.25, 0.5, 1, 2, 4, 8],
            "kernel": ["rbf"],
            "kernel_params": [{"gamma": gamma} for gamma in [0.125, 0.5, 1, 2, 4, 8]]
        },
        {
            "C": [0.25, 0.5, 1, 2, 4, 8],
            "kernel": ["exponential"],
            "kernel_params": [{"gamma": gamma} for gamma in [0.125, 0.5, 1, 2, 4, 8]]
        },
        {
            "C": [0.25, 0.5, 1, 2, 4, 8],
            "kernel": ["laplacian"],
            "kernel_params": [{"gamma": gamma} for gamma in [0.125, 0.5, 1, 2, 4, 8]]
        },
        {
            "C": [0.25, 0.5, 1, 2, 4, 8],
            "kernel": ["linear"],
            "kernel_params": [{"gamma": None}]
        },
        ]

    results = []
    for param_grid in params:
        mean, std = run_experiment(
            n=n,
            censoring_rate=censoring,
            model_class=WSVCR_QP,
            param_grid=param_grid,
            n_sim=100
        )

        results.append(
            {
                "kernel": param_grid["kernel"][0],
                "mean_cindex": mean,
                "std_cindex": std,
                "n": n,
                "prop_cens": censoring
            }
        )
    return results

In [146]:
results_final = []
for cens  in [0.1, 0.25, 0.5]:
    results_final.append(run_simul(50, cens))

### Cox Simulation Data

In [210]:
def train_cox_model(X_train, T_train, d_train, X_val, T_val, d_val, param_grid):

    best_score = -np.inf
    best_model = None

    y_train = Surv.from_arrays(d_train.astype(bool), T_train)
    y_val_struct = Surv.from_arrays(d_val.astype(bool), T_val)

    for params in ParameterGrid(param_grid):

        model = CoxPHSurvivalAnalysis(**params)
        model.fit(X_train, y_train)

        pred = model.predict(X_val)
        score = cindex_survival(T_val, -pred, d_val)

        if score > best_score:
            best_score = score
            best_model = model

    return best_model

In [211]:
def run_experiment_cox(n, censoring_rate, param_grid, n_sim=50):

    scores = []

    for _ in range(n_sim):

        df = generate_wsvcr_data(n=n, censoring_rate=censoring_rate)

        X = df[["X1","X2","X3"]].values
        T = df["T"].values
        delta = df["delta"].values

        X_train, X_test, T_train, T_test, d_train, d_test = train_test_split(
            X, T, delta,
            test_size=0.3,
            stratify=delta
        )

        X_tr, X_val, T_tr, T_val, d_tr, d_val = train_test_split(
            X_train, T_train, d_train,
            test_size=0.3,
            stratify=d_train
        )

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)

        best_model = train_cox_model(
            X_tr, T_tr, d_tr,
            X_val, T_val, d_val,
            param_grid
        )

        pred_test = best_model.predict(X_test)
        score = cindex_survival(T_test, -pred_test, d_test)

        scores.append(score)

    return np.mean(scores), np.std(scores)

In [220]:
param_grid_cox = {
    # "alpha": [0.0, 0.01, 0.1, 1.0]  # regularização L2
    "alpha": [0]
}

def run_simul_cox():

    results = []

    scenarios = [
        (50, [0.1, 0.25, 0.5]),
        (100, [0.1, 0.25, 0.5]),
        (300, [0.1, 0.25, 0.5, 0.95])
    ]

    for n, censor_list in scenarios:
        for cens in censor_list:

            mean, std = run_experiment_cox(
                n=n,
                censoring_rate=cens,
                param_grid=param_grid_cox,
                n_sim=100
            )

            results.append({
                "model": "CoxPH",
                "mean_cindex": mean,
                "std_cindex": std,
                "n": n,
                "prop_cens": cens
            })

            print(f"Cox | n={n} | cens={cens} -> {mean:.3f} ± {std:.3f}")

    return pd.DataFrame(results)


In [219]:
df_new = pd.concat([
    pd.read_csv("simul_cox_data.csv"),
    pd.read_csv("simul_data_50_n.csv"),
    pd.read_csv("simul_data_100_n.csv"),
    pd.read_csv("simul_data_300_n.csv")
], ignore_index=True).query("prop_cens != 0.95")

idx = df_new.groupby(['kernel', 'n', 'prop_cens'])['mean_cindex'].idxmax()
df_best_new = df_new.loc[idx]

styled_new = df_best_new.pivot_table(
    index='kernel',
    columns=['n', 'prop_cens'],
    values='mean_cindex'
)

# 4. Add Mean and Sort
styled_new['Mean'] = styled_new.mean(axis=1)
styled_new = styled_new.sort_values(by='Mean', ascending=False)

print(styled_new.round(3))

n               50                  100                  300                \
prop_cens      0.1   0.25    0.5    0.1   0.25    0.5    0.1   0.25    0.5   
kernel                                                                       
laplacian    0.730  0.751  0.777  0.771  0.784  0.795  0.825  0.833  0.848   
cauchy       0.739  0.754  0.774  0.771  0.773  0.799  0.817  0.825  0.846   
poly         0.734  0.757  0.779  0.776  0.783  0.813  0.798  0.808  0.842   
exponential  0.730  0.736  0.763  0.772  0.788  0.791  0.819  0.827  0.848   
rbf          0.720  0.726  0.760  0.769  0.782  0.783  0.809  0.815  0.836   
wavelet      0.709  0.718  0.723  0.758  0.769  0.783  0.794  0.801  0.821   
CoxPH        0.727  0.736  0.782  0.750  0.755  0.793  0.743  0.766  0.806   
linear       0.704  0.742  0.755  0.730  0.750  0.782  0.748  0.764  0.804   

n             Mean  
prop_cens           
kernel              
laplacian    0.791  
cauchy       0.789  
poly         0.788  
exponential  0.

### Real Data Application

In [30]:
# pd.read_csv("../data/cgd.csv")
# pd.read_csv("../data/lung.csv")
# pd.read_csv("../data/ovarian.csv")
# pd.read_csv("../data/pbc.csv")
# pd.read_csv("../data/rats2.csv")
# pd.read_csv("../data/retinopathy.csv")

In [222]:
# import numpy as np
# import pandas as pd

# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import train_test_split

# from sksurv.datasets import (
#     load_veterans_lung_cancer,
#     load_whas500,
#     load_gbsg2,
#     # load_flchain
# )

# from sksurv.metrics import concordance_index_censored


# def load_survival_dataset(loader):
#     X_raw, y_struct = loader()

#     X = pd.DataFrame(X_raw)

#     X = pd.get_dummies(X, drop_first=True)

#     event_col, time_col = y_struct.dtype.names

#     df = X.copy()
#     df["event"] = y_struct[event_col].astype(int)
#     df["time"] = y_struct[time_col]

#     return df


# def to_interval_format(df):
#     T = df["time"].values
#     delta = df["event"].values

#     l = T.copy()
#     u = np.where(delta == 1, T, np.inf)

#     X = df.drop(columns=["time", "event"]).values

#     return X, l, u


# def train_test_eval(df, C, A, kernel):

#     X, l, u = to_interval_format(df)

#     X_train, X_test, l_train, l_test, u_train, u_test = train_test_split(
#         X, l, u,
#         test_size=0.3
#     )

#     scaler = StandardScaler()
#     X_train = scaler.fit_transform(X_train)
#     X_test = scaler.transform(X_test)

#     params = {"A": A} if kernel == "wavelet" else {"gamma": A}

#     model = WSVCR_QP(
#         C=C,
#         kernel=kernel,
#         kernel_params=params
#     )

#     model.fit(X_train, l_train, u_train)

#     y_pred = model.predict(X_test)

#     risk = -y_pred

#     event_test = np.isfinite(u_test)
#     time_test = l_test

#     cindex = concordance_index_censored(
#         event_test,
#         time_test,
#         risk
#     )[0]

#     return cindex


# def grid_search_real(df, n_repeats=10):

#     C_values = [0.25, 0.5, 1, 2, 4]
#     A_values = [0.125, 0.25, 0.5, 1, 2, 4, 8]

#     kernels = [
#         "wavelet",
#         "linear",
#         "rbf",
#         "exponential",
#         "poly",
#         "cauchy",
#         "laplacian"
#     ]

#     results = []

#     for kernel in kernels:
#         for C in C_values:
#             for A in A_values:

#                 scores = []

#                 for _ in range(n_repeats):
#                     try:
#                         score = train_test_eval(df, C, A, kernel)
#                         scores.append(score)
#                     except Exception as e:
#                         print(f"{kernel} failed: {e}")

#                 if len(scores) > 0:
#                     mean_score = np.mean(scores)
#                     std_score = np.std(scores)

#                     print(kernel, C, A, mean_score)

#                     results.append({
#                         "kernel": kernel,
#                         "C": C,
#                         "A": A if kernel == "wavelet" else None,
#                         "gamma": A if kernel != "wavelet" else None,
#                         "mean_cindex": mean_score,
#                         "std": std_score
#                     })

#     return pd.DataFrame(results)


# datasets = {
#     "veterans": load_veterans_lung_cancer,
#     "whas500": load_whas500,
#     "gbsg2": load_gbsg2,
#     # "flchain": load_flchain
# }

# all_results = []

# for dataset_name, loader in datasets.items():
#     print(f"\n{'='*50}")
#     print(f"DATASET: {dataset_name}")
#     print(f"{'='*50}")

#     df = load_survival_dataset(loader)

#     results = grid_search_real(df, n_repeats=10)
#     results["dataset"] = dataset_name

#     all_results.append(results)


# final_results = pd.concat(all_results, ignore_index=True)

# print("\nTOP RESULTS OVERALL:")
# print(
#     final_results
#     .sort_values("mean_cindex", ascending=False)
#     .head(20)
# )

# print("\nBEST PER DATASET:")
# print(
#     final_results
#     .sort_values("mean_cindex", ascending=False)
#     .groupby("dataset")
#     .head(5)
# )


In [227]:
# import numpy as np
# import pandas as pd

# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import train_test_split

# from sksurv.datasets import (
#     load_veterans_lung_cancer,
#     load_whas500,
#     load_gbsg2
# )

# from sksurv.metrics import concordance_index_ipcw
# from sksurv.util import Surv


# # =====================================================
# # LOAD sksurv DATASETS
# # =====================================================
# def load_survival_dataset(loader):
#     X_raw, y_struct = loader()

#     X = pd.DataFrame(X_raw)
#     X = pd.get_dummies(X, drop_first=True)

#     event_col, time_col = y_struct.dtype.names

#     df = X.copy()
#     df["event"] = y_struct[event_col].astype(int)
#     df["time"] = y_struct[time_col]

#     return df


# def load_csv_survival(path, name):
#     df = pd.read_csv(path)

#     if name == "cgd":
#         df["time"] = df["tstop"]
#         df["event"] = df["status"]

#     elif name == "lung":
#         df["time"] = df["time"]
#         df["event"] = (df["status"] == 2).astype(int)

#     elif name == "ovarian":
#         df["time"] = df["futime"]
#         df["event"] = df["fustat"]

#     elif name == "pbc":
#         df["time"] = df["time"]
#         df["event"] = (df["status"] == 2).astype(int)

#     elif name == "rats2":
#         df["time"] = df["time2"]
#         df["event"] = df["status"]

#     elif name == "retinopathy":
#         df["time"] = df["futime"]
#         df["event"] = df["status"]

#     else:
#         raise ValueError(f"Dataset {name} not configured.")

#     df = df.dropna()

#     drop_cols = [col for col in df.columns if col.lower() in ["id", "obs"]]
#     df = df.drop(columns=drop_cols, errors="ignore")

#     cols = [c for c in df.columns if c not in ["time", "event"]]
#     df = df[cols + ["time", "event"]]

#     df = pd.get_dummies(df, drop_first=True)

#     return df


# # =====================================================
# # INTERVAL FORMAT
# # =====================================================
# def to_interval_format(df):
#     T = df["time"].values
#     delta = df["event"].values

#     l = T.copy()
#     u = np.where(delta == 1, T, np.inf)

#     X = df.drop(columns=["time", "event"]).values

#     return X, l, u, T, delta


# # =====================================================
# # WSVCR EVALUATION (FIXED IPCW)
# # =====================================================
# def train_test_eval(df, C, A, kernel):

#     X, l, u, T, delta = to_interval_format(df)

#     X_train, X_test, l_train, l_test, u_train, u_test, T_train, T_test, d_train, d_test = train_test_split(
#         X, l, u, T, delta,
#         test_size=0.3,
#         stratify=delta if len(np.unique(delta)) > 1 else None,
#         random_state=None
#     )

#     scaler = StandardScaler()
#     X_train = scaler.fit_transform(X_train)
#     X_test  = scaler.transform(X_test)

#     params = {"A": A} if kernel == "wavelet" else {"gamma": A}

#     model = WSVCR_QP(
#         C=C,
#         kernel=kernel,
#         kernel_params=params
#     )

#     model.fit(X_train, l_train, u_train)

#     y_pred = model.predict(X_test)

#     # =========================
#     # IPCW FIX
#     # =========================
#     y_train_struct = Surv.from_arrays(d_train.astype(bool), T_train)
#     y_test_struct  = Surv.from_arrays(d_test.astype(bool), T_test)

#     max_train_time = T_train.max()

#     mask = T_test <= max_train_time

#     # evita erro quando sobra poucos pontos
#     if mask.sum() < 5:
#         raise ValueError("Too few valid test samples after IPCW filtering")

#     y_test_struct = y_test_struct[mask]
#     y_pred = y_pred[mask]

#     cindex = concordance_index_ipcw(
#         y_train_struct,
#         y_test_struct,
#         y_pred
#     )[0]

#     return cindex


# # =====================================================
# # GRID SEARCH
# # =====================================================
# def grid_search_real(df, n_repeats=10):

#     C_values = [0.25, 0.5, 1, 2, 4]
#     A_values = [0.125, 0.25, 0.5, 1, 2, 4, 8]

#     kernels = [
#         "wavelet",
#         "linear",
#         "rbf",
#         "exponential",
#         "poly",
#         "cauchy",
#         "laplacian"
#     ]

#     results = []

#     for kernel in kernels:
#         for C in C_values:
#             for A in A_values:

#                 scores = []

#                 for _ in range(n_repeats):
#                     try:
#                         score = train_test_eval(df, C, A, kernel)
#                         scores.append(score)
#                     except Exception as e:
#                         print(f"{kernel} failed: {e}")

#                 if scores:
#                     results.append({
#                         "kernel": kernel,
#                         "C": C,
#                         "A": A if kernel == "wavelet" else None,
#                         "gamma": A if kernel != "wavelet" else None,
#                         "mean_cindex": np.mean(scores),
#                         "std": np.std(scores)
#                     })

#                     print(kernel, C, A, np.mean(scores))

#     return pd.DataFrame(results)


# # =====================================================
# # DATASETS
# # =====================================================
# datasets_sksurv = {
#     "veterans": load_veterans_lung_cancer,
#     # "whas500": load_whas500,
#     # "gbsg2": load_gbsg2,
# }

# datasets_csv = {
#     "cgd": "../data/cgd.csv",
#     "lung": "../data/lung.csv",
#     "ovarian": "../data/ovarian.csv",
#     # "pbc": "../data/pbc.csv",
#     # "rats2": "../data/rats2.csv",
#     # "retinopathy": "../data/retinopathy.csv"
# }


# # =====================================================
# # RUN EXPERIMENTS
# # =====================================================
# all_results = []

# # --- sksurv datasets ---
# for dataset_name, loader in datasets_sksurv.items():
#     print(f"\n{'='*50}")
#     print(f"DATASET: {dataset_name}")
#     print(f"{'='*50}")

#     df = load_survival_dataset(loader)

#     results = grid_search_real(df, n_repeats=10)
#     results["dataset"] = dataset_name

#     all_results.append(results)


# # --- CSV datasets ---
# for dataset_name, path in datasets_csv.items():
#     print(f"\n{'='*50}")
#     print(f"DATASET: {dataset_name}")
#     print(f"{'='*50}")

#     df = load_csv_survival(path, dataset_name)

#     results = grid_search_real(df, n_repeats=10)
#     results["dataset"] = dataset_name

#     all_results.append(results)


# # =====================================================
# # FINAL RESULTS
# # =====================================================
# final_results = pd.concat(all_results, ignore_index=True)

# print("\nTOP RESULTS OVERALL:")
# print(
#     final_results
#     .sort_values("mean_cindex", ascending=False)
#     .head(20)
# )

# print("\nBEST PER DATASET:")
# print(
#     final_results
#     .sort_values("mean_cindex", ascending=False)
#     .groupby("dataset")
#     .head(5)
# )


import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sksurv.datasets import (
    load_veterans_lung_cancer,
    load_whas500,
    load_gbsg2
)

from sksurv.metrics import concordance_index_censored


# =====================================================
# LOAD sksurv DATASETS
# =====================================================
def load_survival_dataset(loader):
    X_raw, y_struct = loader()

    X = pd.DataFrame(X_raw)
    X = pd.get_dummies(X, drop_first=True)

    event_col, time_col = y_struct.dtype.names

    df = X.copy()
    df["event"] = y_struct[event_col].astype(int)
    df["time"] = y_struct[time_col]

    return df

def load_csv_survival(path, name):

    df = pd.read_csv(path)

    # =========================
    # DATASET-SPECIFIC RULES
    # =========================
    if name == "cgd":
        # counting process → use stop time
        df["time"] = df["tstop"]
        df["event"] = df["status"]

    elif name == "lung":
        df["time"] = df["time"]
        df["event"] = (df["status"] == 2).astype(int)  # 2 = death

    elif name == "ovarian":
        df["time"] = df["futime"]
        df["event"] = df["fustat"]

    elif name == "pbc":
        df["time"] = df["time"]
        df["event"] = (df["status"] == 2).astype(int)  # 2 = death

    elif name == "rats2":
        # interval data → approximate using right endpoint
        df["time"] = df["time2"]
        df["event"] = df["status"]

    elif name == "retinopathy":
        df["time"] = df["futime"]
        df["event"] = df["status"]

    else:
        raise ValueError(f"Dataset {name} not configured.")

    # =========================
    # CLEANING
    # =========================
    df = df.dropna()

    # remove obvious ID columns
    drop_cols = [col for col in df.columns if col.lower() in ["id", "obs"]]
    df = df.drop(columns=drop_cols, errors="ignore")

    # keep only features + time/event
    cols = [c for c in df.columns if c not in ["time", "event"]]
    df = df[cols + ["time", "event"]]

    # one-hot encode categoricals
    df = pd.get_dummies(df, drop_first=True)

    return df


# =====================================================
# INTERVAL FORMAT
# =====================================================
def to_interval_format(df):
    T = df["time"].values
    delta = df["event"].values

    l = T.copy()
    u = np.where(delta == 1, T, np.inf)

    X = df.drop(columns=["time", "event"]).values

    return X, l, u


# =====================================================
# WSVCR EVALUATION
# =====================================================
def train_test_eval(df, C, A, kernel):

    X, l, u = to_interval_format(df)

    X_train, X_test, l_train, l_test, u_train, u_test = train_test_split(
        X, l, u, test_size=0.3
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    params = {"A": A} if kernel == "wavelet" else {"gamma": A}

    model = WSVCR_QP(
        C=C,
        kernel=kernel,
        kernel_params=params
    )

    model.fit(X_train, l_train, u_train)

    y_pred = model.predict(X_test)

    risk = -y_pred

    event_test = np.isfinite(u_test)
    time_test = l_test

    cindex = concordance_index_censored(
        event_test,
        time_test,
        risk
    )[0]

    return cindex


# =====================================================
# GRID SEARCH
# =====================================================
def grid_search_real(df, n_repeats=10):

    C_values = [0.25, 0.5, 1, 2, 4]
    A_values = [0.125, 0.25, 0.5, 1, 2, 4, 8]

    kernels = [
        "wavelet",
        "linear",
        "rbf",
        "exponential",
        "poly",
        "cauchy",
        "laplacian"
    ]

    results = []

    for kernel in kernels:
        for C in C_values:
            for A in A_values:

                scores = []

                for _ in range(n_repeats):
                    try:
                        score = train_test_eval(df, C, A, kernel)
                        scores.append(score)
                    except Exception as e:
                        print(f"{kernel} failed: {e}")

                if scores:
                    mean_score = np.mean(scores)
                    std_score = np.std(scores)

                    print(kernel, C, A, mean_score)

                    results.append({
                        "kernel": kernel,
                        "C": C,
                        "A": A if kernel == "wavelet" else None,
                        "gamma": A if kernel != "wavelet" else None,
                        "mean_cindex": mean_score,
                        "std": std_score
                    })

    return pd.DataFrame(results)


# =====================================================
# DATASETS
# =====================================================
datasets_sksurv = {
    "veterans": load_veterans_lung_cancer,
    # "whas500": load_whas500,
    # "gbsg2": load_gbsg2,
}

datasets_csv = {
    # "cgd": "../data/cgd.csv",
    "lung": "../data/lung.csv",
    "ovarian": "../data/ovarian.csv",
    "pbc": "../data/pbc.csv",
    # "rats2": "../data/rats2.csv",
    # "retinopathy": "../data/retinopathy.csv"
}


# =====================================================
# RUN EXPERIMENTS
# =====================================================
all_results = []

# --- sksurv datasets ---
for dataset_name, loader in datasets_sksurv.items():
    print(f"\n{'='*50}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*50}")

    df = load_survival_dataset(loader)

    results = grid_search_real(df, n_repeats=10)
    results["dataset"] = dataset_name

    all_results.append(results)


# --- CSV datasets ---
for dataset_name, path in datasets_csv.items():
    print(f"\n{'='*50}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*50}")

    df = load_csv_survival(path, dataset_name)

    results = grid_search_real(df, n_repeats=10)
    results["dataset"] = dataset_name

    all_results.append(results)


# =====================================================
# FINAL RESULTS
# =====================================================
final_results = pd.concat(all_results, ignore_index=True)

print("\nTOP RESULTS OVERALL:")
print(
    final_results
    .sort_values("mean_cindex", ascending=False)
    .head(20)
)

print("\nBEST PER DATASET:")
print(
    final_results
    .sort_values("mean_cindex", ascending=False)
    .groupby("dataset")
    .head(5)
)


DATASET: veterans
wavelet 0.25 0.125 0.5163596739936404
wavelet 0.25 0.25 0.506614920511727
wavelet 0.25 0.5 0.46389223329710144
wavelet 0.25 1 0.5323670490065712
wavelet 0.25 2 0.5992186280440117
wavelet 0.25 4 0.6834247439850907
wavelet 0.25 8 0.678889248079747
wavelet 0.5 0.125 0.5041633837259714
wavelet 0.5 0.25 0.4959977872764922
wavelet 0.5 0.5 0.45699805287979717
wavelet 0.5 1 0.5417469920572385
wavelet 0.5 2 0.5876308209860428
wavelet 0.5 4 0.6931792480590842
wavelet 0.5 8 0.684975910753996
wavelet 1 0.125 0.47934198321047783
wavelet 1 0.25 0.48123690227419036
wavelet 1 0.5 0.473905737878683
wavelet 1 1 0.5449371111619845
wavelet 1 2 0.5864754762975919
wavelet 1 4 0.6717159490564302
wavelet 1 8 0.6797571338416647
wavelet 2 0.125 0.5503349248704181
wavelet 2 0.25 0.5113735590944221
wavelet 2 0.5 0.4830209270007745
wavelet 2 1 0.5840574013586564
wavelet 2 2 0.5894831277660032
wavelet 2 4 0.6879105506021548
wavelet 2 8 0.6900831428324767
wavelet 4 0.125 0.5145780515411609
wavelet

### Cox in Real data

In [244]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sksurv.datasets import (
    load_veterans_lung_cancer,
    load_whas500,
    load_gbsg2
)

from sksurv.metrics import concordance_index_censored
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.util import Surv


# =====================================================
# LOAD sksurv DATASETS
# =====================================================
def load_survival_dataset(loader):
    X_raw, y_struct = loader()

    X = pd.DataFrame(X_raw)
    X = pd.get_dummies(X, drop_first=True)

    event_col, time_col = y_struct.dtype.names

    df = X.copy()
    df["event"] = y_struct[event_col].astype(int)
    df["time"] = y_struct[time_col]

    return df


# =====================================================
# LOAD CSV DATASETS (R survival package)
# =====================================================
def load_csv_survival(path, name):

    df = pd.read_csv(path)

    if name == "cgd":
        df["time"] = df["tstop"]
        df["event"] = df["status"]

    elif name == "lung":
        df["time"] = df["time"]
        df["event"] = (df["status"] == 2).astype(int)

    elif name == "ovarian":
        df["time"] = df["futime"]
        df["event"] = df["fustat"]

    elif name == "pbc":
        df["time"] = df["time"]
        df["event"] = (df["status"] == 2).astype(int)

    elif name == "rats2":
        df["time"] = df["time2"]
        df["event"] = df["status"]

    elif name == "retinopathy":
        df["time"] = df["futime"]
        df["event"] = df["status"]

    else:
        raise ValueError(f"Dataset {name} not configured.")

    df = df.dropna()

    drop_cols = [col for col in df.columns if col.lower() in ["id", "obs"]]
    df = df.drop(columns=drop_cols, errors="ignore")

    cols = [c for c in df.columns if c not in ["time", "event"]]
    df = df[cols + ["time", "event"]]

    df = pd.get_dummies(df, drop_first=True)

    return df


# =====================================================
# COX EVALUATION
# =====================================================
def train_test_eval_cox(df):

    X = df.drop(columns=["time", "event"]).values
    time = df["time"].values
    event = df["event"].values

    X_train, X_test, t_train, t_test, e_train, e_test = train_test_split(
        X, time, event, test_size=0.3
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    y_train = Surv.from_arrays(
        event=e_train.astype(bool),
        time=t_train
    )

    model = CoxPHSurvivalAnalysis(n_iter=100000)
    model.fit(X_train, y_train)

    # Cox returns risk directly
    risk = model.predict(X_test)

    cindex = concordance_index_censored(
        e_test.astype(bool),
        t_test,
        risk
    )[0]

    return cindex


# =====================================================
# EVALUATE COX (NO GRID SEARCH)
# =====================================================
def evaluate_cox(df, n_repeats=10):

    scores = []

    for _ in range(n_repeats):
        try:
            score = train_test_eval_cox(df)
            scores.append(score)
        except Exception as e:
            print("cox failed:", e)

    return {
        "model": "CoxPH",
        "mean_cindex": np.mean(scores),
        "std": np.std(scores)
    }


# =====================================================
# DATASETS
# =====================================================
datasets_sksurv = {
    "veterans": load_veterans_lung_cancer,
    "whas500": load_whas500,
    "gbsg2": load_gbsg2,
}

datasets_csv = {
    # "cgd": "../data/cgd.csv",
    "lung": "../data/lung.csv",
    # "ovarian": "../data/ovarian.csv",
    # "pbc": "../data/pbc.csv",
    # "rats2": "../data/rats2.csv",
    # "retinopathy": "../data/retinopathy.csv"
}


# =====================================================
# RUN EXPERIMENTS (COX ONLY)
# =====================================================
all_results = []

# --- sksurv datasets ---
for dataset_name, loader in datasets_sksurv.items():
    print(f"\n{'='*50}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*50}")

    df = load_survival_dataset(loader)

    result = evaluate_cox(df, n_repeats=30)
    result["dataset"] = dataset_name

    print("CoxPH:", result["mean_cindex"])

    all_results.append(result)


# --- CSV datasets ---
for dataset_name, path in datasets_csv.items():
    print(f"\n{'='*50}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*50}")

    df = load_csv_survival(path, dataset_name)

    result = evaluate_cox(df, n_repeats=30)
    result["dataset"] = dataset_name

    print("CoxPH:", result["mean_cindex"])

    all_results.append(result)


# =====================================================
# FINAL TABLE
# =====================================================
final_results = pd.DataFrame(all_results)

print("\n===== COX RESULTS =====")
print(final_results.sort_values("mean_cindex", ascending=False))


DATASET: veterans
CoxPH: 0.7110405968271396

DATASET: whas500
CoxPH: 0.7625598497317722

DATASET: gbsg2
CoxPH: 0.6800156134649911

DATASET: lung
CoxPH: 0.7102659566057864

===== COX RESULTS =====
   model  mean_cindex       std   dataset
1  CoxPH     0.762560  0.025943   whas500
0  CoxPH     0.711041  0.037571  veterans
3  CoxPH     0.710266  0.038292      lung
2  CoxPH     0.680016  0.022538     gbsg2


In [ ]:
# pd.concat(
#     [
#         final_results.rename(columns={"model": "kernel"}),
#         pd.read_csv("real_data_results2.csv")
#     ])

,kernel,mean_cindex,std,dataset,C,A,gamma
0,CoxPH,0.711041,0.037571,veterans,NaN,NaN,NaN
1,CoxPH,0.762560,0.025943,whas500,NaN,NaN,NaN
2,CoxPH,0.680016,0.022538,gbsg2,NaN,NaN,NaN
3,CoxPH,0.710266,0.038292,lung,NaN,NaN,NaN
0,wavelet,0.504495,0.006778,cgd,0.25,0.125,NaN
...,...,...,...,...,...,...,...
1465,laplacian,0.916094,0.016689,retinopathy,4.00,NaN,0.5
1466,laplacian,0.895306,0.015462,retinopathy,4.00,NaN,1.0
1467,laplacian,0.882198,0.020137,retinopathy,4.00,NaN,2.0
1468,laplacian,0.879979,0.014374,retinopathy,4.00,NaN,4.0


In [257]:
final_results.dataset.unique()

<StringArray>
['veterans', 'whas500', 'gbsg2', 'lung']
Length: 4, dtype: str

In [263]:
import pandas as pd

df_real = pd.concat([
    final_results.rename(columns={"model": "kernel"}),
    pd.read_csv("real_data_results.csv")
], ignore_index=True).fillna(0).query("dataset != 'lung'")

idx = df_real.groupby(['kernel', 'dataset'])['mean_cindex'].idxmax()
df_best_real = df_real.loc[idx]

table_real = df_best_real.pivot_table(
    index='kernel',
    columns='dataset',
    values='mean_cindex'
)

# 4. Add horizontal Mean and sort
table_real['Mean'] = table_real.mean(axis=1)
table_real = table_real.sort_values(by='Mean', ascending=False)

print(table_real.round(3))

dataset      gbsg2  veterans  whas500   Mean
kernel                                      
linear       0.691     0.740    0.765  0.732
poly         0.682     0.712    0.762  0.719
CoxPH        0.680     0.711    0.763  0.718
exponential  0.674     0.716    0.761  0.717
wavelet      0.681     0.716    0.730  0.709
cauchy       0.671     0.703    0.751  0.709
laplacian    0.681     0.702    0.742  0.708
rbf          0.640     0.706    0.696  0.681


In [ ]:
# pd.DataFrame(results).to_csv("simulate_30_n_cox.csv", index=False)